# 实验二任务5：Nature平台期刊卷信息提取
根据任务要求，本Notebook实现从Nature平台检索特定期刊卷的所有文章信息

导入必要的库

In [4]:
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


定义获取卷文章信息的函数

In [5]:
def get_volume_articles():
    # 读取nature_llm.json文件
    with open('nature_llm.json', 'r', encoding='utf-8') as f:
        journal_data = json.load(f)

    volume_article_data = []
    processed_volumes = set()

    # 构建期刊名到简写的映射字典（若无法从网页获取，可手动补充完整）
    journal_abbr_mapping = {
        
    }

    for journal in journal_data:
        journal_name = journal['journal']
        if journal_name.startswith('Nature') and not journal_name.startswith('Nature Reviews'):
            for paper in journal['papers']:
                volume = paper['volume_page_info'].split(',')[0].split(':')[1].strip()
                journal_abbr = journal_abbr_mapping.get(journal_name, '')

                if (journal_name, volume) not in processed_volumes:
                    processed_volumes.add((journal_name, volume))
                    search_url = f"https://www.nature.com/search?q=&journal={journal_abbr}&volume={volume}&page=1"
                    response = requests.get(search_url)
                    response.raise_for_status()
                    soup = BeautifulSoup(response.text, 'html.parser')

                    volume_papers = []
                    articles = soup.select('li.app-article-list-row__item article.c-card')
                    for article in articles:
                        title_tag = article.find('h3', class_='c-card__title').find('a')
                        title = title_tag.text.strip() if title_tag else "No title"
                        url = title_tag['href'] if title_tag else "No URL"
                        authors = []
                        author_list = article.select('[itemprop="creator"] [itemprop="name"]')
                        for author in author_list:
                            authors.append(author.text.strip())
                        desc_tag = article.find('div', {'data-test': 'article-description'})
                        description = desc_tag.text.strip() if desc_tag else "no description"
                        meta_section = article.find('div', class_='c-meta')
                        article_type = meta_section.find('span', class_='c-meta__type').text.strip() if meta_section else "Unknown"

                        volume_papers.append({
                            "title": title,
                            "authors": authors,
                            "url": url,
                            "description": description,
                            "type": article_type
                        })

                    # 新增条件：仅当papers列表非空时才添加到结果
                    if volume_papers:
                        volume_article_data.append({
                            "journal": journal_name,
                            "volume": volume,
                            "papers": volume_papers
                        })

    # 将数据保存到volume.json文件
    with open('volume.json', 'w', encoding='utf-8') as f:
        json.dump(volume_article_data, f, indent=2, ensure_ascii=False)


调用函数执行获取操作

In [6]:
if __name__ == "__main__":
    get_volume_articles()
